<a href="https://colab.research.google.com/github/sourabh90/mlops-starter/blob/main/1_1_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### ML Pipelines with ZenML

Install ZenML Server, sklearn integration

In [4]:
%pip install "zenml[server]==0.80"
!zenml integration install sklearn -y

import IPython

# auto restart kernel
IPython.Application.instance().kernel.do_shutdown(restart=True)

⠏ Installing integrations...


{'status': 'ok', 'restart': True}

#### Install Python NGROK library and authenticate
NGROK will enable visualization

In [1]:
NGROK_TOKEN = '34CncgUkecxJtj8QgSAJ55Wy2br_4Awb6U7zaQnzWEBGPqida'

In [2]:
from zenml.environment import Environment

if Environment.in_google_colab():
    !pip install pyngrok
    !ngrok authtoken {NGROK_TOKEN}

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


#### Setup ZenML

In [3]:
# Remove any previous ZenML setup or config
!rm -rf .zen
!zenml init

The ZenML global configuration version (0.90.0) is higher than the version of ZenML currently being used (0.80.0). Read more about this issue and how to solve it here: https://docs.zenml.io/reference/global-settings#version-mismatch-downgrading
⠋ Initializing ZenML repository at /content.
⠙ Initializing ZenML repository at /content.
⠼ Initializing ZenML repository at /content.
⠴ Initializing ZenML repository at /content.
⠦ Initializing ZenML repository at /content.
⠧ Initializing ZenML repository at /content.
⠇ Initializing ZenML repository at /content.
⠏ Initializing ZenML repository at /content.
⠋ Initializing ZenML repository at /content.
⠙ Initializing ZenML repository at /content.

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/local/lib/python3.12/dist-packages/alembic/script/base.py:233 in        │
│ _catch_revision_errors                                                       │
│                                                            

#### Example ML Experimentation Code

In [8]:
import numpy as np
from sklearn.base import ClassifierMixin
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def train_test():
  '''Train and test a Scikei-learn SVC classifier on digits'''
  digits = load_digits()
  data = digits.images.reshape((len(digits.images), -1))
  X_train, X_test, y_train, y_test = train_test_split(
      data, digits.target, test_size=0.2, shuffle=False
  )
  model = SVC(gamma=0.001)
  model.fit(X_train, y_train)
  test_acc = model.score(X_test, y_test)
  print(f'Test accuracy: {test_acc}')

train_test()

Test accuracy: 0.9583333333333334


#### Turning ML into pipelines with ZenML


    Zen ML Repository
    
    First Pipeline
    
    Importer --> SVC Trainer --> Evaluator


We can identify 3 distinct steps in out example: data loading, model training and model evaluation. Let's define each of these steps as a ZenML pipeline step simply by moving each step to its own function and decorating them with ZenML's python decorator @step.



In [9]:
from zenml import step
from typing_extensions import Annotated
import pandas as pd
from typing import Tuple

@step
def importer() -> Tuple[
  Annotated[np.ndarray, 'X_train'],
  Annotated[np.ndarray, 'X_test'],
  Annotated[np.ndarray, 'y_train'],
  Annotated[np.ndarray, 'y_test'],
]:
  '''Load the digits datasets as numpy array'''
  digits = load_digits()
  data = digits.images.reshape((len(digits.images), -1))
  X_train, X_test, y_train, y_test = train_test_split(
      data, digits.target, test_size=0.2, shuffle=False
  )
  return X_train, X_test, y_train, y_test


@step
def svc_trainer(
  X_train: np.ndarray,
  y_train: np.ndarray,
) -> ClassifierMixin:
  '''Train an sklearn SVC classifier'''
  model = SVC(gamma=0.001)
  model.fit(X_train, y_train)
  return model


@step
def evaluator(
  X_test: np.ndarray,
  y_test: np.ndarray,
  model: ClassifierMixin
) -> float:
  '''Calculate the test set accuracy of an sklearn model'''
  test_acc = model.score(X_test, y_test)
  print(f'Test accuracy: {test_acc}')
  return test_acc



Similarly we can use ZenML's @pipeline decorator to connect all of our steps into an MLOps pipeline.

In [10]:
from zenml import pipeline

@pipeline
def digits_pipeline():
  '''Links all the steps together in a MLOps pipeline'''
  X_train, X_test, y_train, y_test = importer()
  model = svc_trainer(X_train, y_train)
  evaluator(X_test, y_test, model)


#### Running ZenML Pipelines


In [14]:
digits_svc_pipeline = digits_pipeline()

INFO:zenml.pipelines.pipeline_definition:Initiating a new run for the pipeline: `digits_pipeline`.


Initiating a new run for the pipeline: digits_pipeline.


DEBUG:zenml.integrations.integration:Requirement 'label-studio-sdk>=1.0.0' for integration 'label_studio' is not installed or installed with the wrong version.
DEBUG:zenml.integrations.registry:Integration `label_studio` could not be activated.
DEBUG:zenml.integrations.integration:Requirement 'kubernetes==18.20.0' for integration 'seldon' is not installed or installed with the wrong version.
DEBUG:zenml.integrations.registry:Integration `seldon` could not be activated.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <cell line: 0>:1                                                                              │
│                                                                                                  │
│ ❱ 1 digits_svc_pipeline = digits_pipeline()                                                      │
│   2                                                                                              │
│                                                                                                  │
│ /usr/local/lib/python3.12/dist-packages/zenml/pipelines/pipeline_definition.py:1599 in __call__  │
│                                                                                                  │
│   1596 │   │   │   return self.entrypoint(*args, **kwargs)  # type: ignore[no-any-return]        │
│   1597 │   │                                                                                     │
│   1598 │   │   self.prepare(*args, **kwargs)                                                     │
│ ❱ 1599 │   │   return self._run()                                                                │
│   1600 │                                                                                         │
│   1601 │   def _call_entrypoint(self, *args: Any, **kwargs: Any) -> None:                        │
│   1602 │   │   """Calls the pipeline entrypoint function with the given arguments.               │
│                                                                                                  │
│ /usr/local/lib/python3.12/dist-packages/zenml/pipelines/pipeline_definition.py:986 in _run       │
│                                                                                                  │
│    983 │   │   │   │   )                                                                         │
│    984 │   │   │                                                                                 │
│    985 │   │   │   with logs_context:                                                            │
│ ❱  986 │   │   │   │   snapshot = self._create_snapshot(**self._run_args)                        │
│    987 │   │   │   │                                                                             │
│    988 │   │   │   │   self.log_pipeline_snapshot_metadata(snapshot)                             │
│    989 │   │   │   │   run = (                                                                   │
│                                                                                                  │
│ /usr/local/lib/python3.12/dist-packages/zenml/pipelines/pipeline_definition.py:786 in            │
│ _create_snapshot                                                                                 │
│                                                                                                  │
│    783 │   │   │   ValueError: If the orchestrator doesn't support scheduling, but a             │
│    784 │   │   │   │   schedule was given                                                        │
│    785 │   │   """                                                                               │
│ ❱  786 │   │   snapshot, schedule, build = self._compile(                                        │
│    787 │   │   │   config_path=config_path,                                                      │
│    788 │   │   │   run_name=run_name,                                                            │
│    789 │   │   │   enable_cache=enable_cache,                                                    │
│                                                                                                  │
│ /usr/local/lib/python3.12/dist-packages/zenml/pipelines/pipeline_definition.py:1229 in _compile  │
│                                                                                                  │
│   1226 │   │   # Activating the built-in integrations to lo

After running the pipeline you can visualize the dashboard.

In [15]:
from zenml.environment import Environment

def start_zenml_dashboard(port=8237):
  if Environment.in_google_colab():
    from pyngrok import ngrok

    public_url = ngrok.connect(port)
    print(f'\x1b[31mIn Colab, use this URL instead: {public_url}!\x1b[0m')
    !zenml up --blocking --port {port}

  else:
    !zenml up --port {port}

start_zenml_dashboard()


INFO:pyngrok.ngrok:Opening tunnel named: http-8237-a91f2bbe-f2fb-4394-addd-d1586681075f


Opening tunnel named: http-8237-a91f2bbe-f2fb-4394-addd-d1586681075f


INFO:pyngrok.process.ngrok:t=2025-10-19T17:55:13+0000 lvl=info msg="no configuration paths supplied"


t=2025-10-19T17:55:13+0000 lvl=info msg="no configuration paths supplied"


INFO:pyngrok.process.ngrok:t=2025-10-19T17:55:13+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml


...

t=2025-10-19T17:55:13+0000 lvl=info msg=start pg=/api/tunnels id=3dc390c2d3558493


INFO:pyngrok.process.ngrok:t=2025-10-19T17:55:13+0000 lvl=info msg="started tunnel" obj=tunnels name=http-8237-a91f2bbe-f2fb-4394-addd-d1586681075f addr=http://localhost:8237 url=https://nonadvantageously-unoverdrawn-neely.ngrok-free.dev


In Colab, use this URL instead: NgrokTunnel: "https://nonadvantageously-unoverdrawn-neely.ngrok-free.dev" -> "http://localhost:8237"!
t=2025-10-19T17:55:13+0000 lvl=info msg="started tunnel" obj=tunnels name=http-8237-a91f2bbe-f2fb-4394-addd-d1586681075f addr=http://localhost:8237 url=https://nonadvantageously-unoverdrawn-neely.ngrok-free.dev


INFO:pyngrok.process.ngrok:t=2025-10-19T17:55:13+0000 lvl=info msg=end pg=/api/tunnels id=3dc390c2d3558493 status=201 dur=135.582224ms


t=2025-10-19T17:55:13+0000 lvl=info msg=end pg=/api/tunnels id=3dc390c2d3558493 status=201 dur=135.582224ms
The `zenml up` command is deprecated and will be removed in a future release. 
Please use the `zenml login --local` command instead.
Calling `zenml login --local`...
The local ZenML dashboard is about to deploy in a blocking process.
Deploying a local daemon ZenML server.
Not writing the global configuration to disk in a ZenML server environment.
Initializing the ZenML global configuration version to 0.90.0
Not writing the global configuration to disk in a ZenML server environment.
Starting ZenML Server as blocking process... press CTRL+C once to stop it.
INFO:     Started server process [12725]
INFO:     Waiting for application startup.
Not writing the global configuration to disk in a ZenML server environment.
Not writing the global configuration to disk in a ZenML server environment.
ERROR:    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/s